Richard Davies · **Automated Data Visualisation for Policymaking** · PP434 · 2026/27

# Cleaning messy data: the UK gilt market

> ⭐ **Stretch example.** This is a longer, harder notebook than the others this week. Try it once you're comfortable with DS5_1 and DS5_2.

This is **Example 1** from the lecture. The UK government borrows by selling bonds called **gilts**. The [Debt Management Office (DMO)](https://dmo.gov.uk/data/gilt-market/gilts-in-issue/) publishes a list of every gilt in issue, but each gilt's details are squashed into one piece of text:

| Name (messy) | Coupon (% interest a year) | Type | Repaid in |
|---|---|---|---|
| `0 1/8% Treasury Gilt 2023` | 0.125 | Treasury Gilt | 2023 |
| `4¼% Treasury Stock 2032` | 4.25 | Treasury Stock | 2032 |
| `0¾% Index-linked Treasury Gilt 2047` | 0.75 | Index-linked Treasury Gilt | 2047 |

We'll turn the messy column into clean ones. As the lecture says, cleaning is often a matter of:
1. **Spotting patterns**: where in the messy text is the number we want?
2. Using **string functions** to split and change the text, with **loops** and **`if` statements** to handle each case.

> **Before you start:** click **File → Save a copy in Drive**. Otherwise, your changes will not be saved.

In [ ]:
# Colab comes with an older version of Altair (v5). This updates it to the latest version (v6),
# so the charts we save match the Vega-Lite version used on our websites.
# Run this before importing Altair. (If you've already imported it, use Runtime → Restart session, then run again.)
# On your own computer (see our local Python guide), this just checks you have the latest version.
%pip install altair --upgrade --quiet

In [ ]:
import pandas as pd     # tables of data
import altair as alt    # charts

<br><br>

---

<br>

## 1. Load the raw data

This is the DMO's list on 20 October 2022, saved as a CSV file. Two things to know about it:
- The first 8 lines are titles and blank lines, so we skip them with `skiprows=8`.
- It was saved by Excel on Windows, so symbols like `£` and `¼` use an older text **encoding**, `cp1252`. Without `encoding="cp1252"`, Python can't read them.

In [ ]:
url = "https://raw.githubusercontent.com/RDeconomist/RDeconomist.github.io/main/data/GiltsInIssueRaw.csv"

df = pd.read_csv(url, skiprows=8, encoding="cp1252")
df.head(12)

<br>

The column names are long and messy too. We give them short, simple names (in the same order):

In [ ]:
df.columns = ["name", "code", "redemptionDate", "firstIssueDate", "dividendDates",
              "nextExDividendDate", "amount", "baseRPI", "amountWithUplift"]
df.head(12)

<br>

Look at the rows: as well as the gilts, there are **section titles** (`Ultra-Short`, `Short`, `Medium`, `Long`, and later two `Index-linked Gilts` headings), blank rows, repeated column headings and notes at the end. We need to deal with all of these.

<br><br>

---

<br>

## 2. A loop that remembers which section it's in

The section titles hold useful information: they group the gilts by how long until they're repaid. But it's in the wrong place: a row of its own, *above* the gilts.

A loop can fix this. It goes down the rows in order and **remembers** the latest section title it has seen:
- **if** a row is a section title (it has no ISIN code, or it's a repeated heading), store its name in `section`;
- every row gets the current `section` added to a list.

`zip()` lets us loop over two columns side by side, getting one `name` and one `code` each time.

In [ ]:
section = None      # no section yet
sections = []       # one entry for every row

for name, code in zip(df["name"], df["code"]):
    if pd.isna(code) or code == "ISIN Code":       # pd.isna() is True for an empty cell
        if pd.notna(name):                          # skip completely blank rows
            section = name.replace("\n", " ")       # some titles have a line break inside
    sections.append(section)

df["section"] = sections
df.head(12)

<br><br>

---

<br>

## 3. Keep only the gilts

Now we can drop the rows that aren't gilts. Spot the pattern: every gilt has an **ISIN code** (an international ID for bonds), and UK ones start with `GB`. So we keep a row **if** its code starts with `GB`:

In [ ]:
print(f"Rows before: {len(df)}")

df = df[df["code"].str.startswith("GB", na=False)].copy()   # na=False: treat empty codes as 'no'

print(f"Rows after: {len(df)}")
df["section"].value_counts()

<br><br>

---

<br>

## 4. Clean one name first

As with loops, get it working for **one** item first. The pattern in every name is:

`[coupon]%` `[type]` `[year]`

So:
- **split** at the `%`: the coupon is before it, the rest after it;
- the **year** is always the last 4 characters;
- the **type** is everything before the year, with spaces **trimmed** off the ends with `.strip()`.

In [ ]:
name = "0 1/8% Treasury Gilt 2023"

coupon_text, rest = name.split("%")      # two pieces: before and after the %
rest = rest.strip()                      # remove spaces at the start and end

year = int(rest[-4:])                    # the last 4 characters, as a whole number
gilt_type = rest[:-4].strip()            # everything except the last 4 characters

print(f"Coupon text: '{coupon_text}'")
print(f"Type: '{gilt_type}'")
print(f"Year: {year}")

<br>

### The coupon: from text to a number

The coupon text is the hardest part. It comes in several forms: `0 1/8`, `4¼`, `1`, and even `1¼ ` (with a space). Two steps:
1. **Replace** the special fraction symbols (`¼`, `½`, `¾`) with ordinary fractions (` 1/4`, ...).
2. **Split** at the spaces, and add up the parts. A **loop** goes through the parts, and an **`if`** checks whether each part is a fraction.

We put these steps in a **function**, so we can reuse them for every gilt:

In [ ]:
def coupon_to_number(text):
    # 1. Swap the special fraction symbols for ordinary fractions
    text = text.replace("¼", " 1/4").replace("½", " 1/2").replace("¾", " 3/4")

    # 2. Add up the parts, e.g. '0 1/8' -> 0 + 1/8 = 0.125
    total = 0
    for part in text.split():                  # split() with nothing in the brackets splits at spaces
        if "/" in part:                        # a fraction, like '1/8'
            top, bottom = part.split("/")
            total = total + int(top) / int(bottom)
        else:                                  # a whole number, like '4'
            total = total + int(part)
    return total


for text in ["0 1/8", "4¼", "1", "1¼ "]:
    print(f"'{text}' becomes {coupon_to_number(text)}")

<br><br>

---

<br>

## 5. Clean every name, with a loop

Now we repeat the steps for every gilt. Each result goes into a list (start empty, then `.append()`), and the lists become new columns.

We also add a **category**, with `if`, `elif` and `else`. Notice `.lower()`: one gilt in the data says `Index-Linked` instead of `Index-linked`. Comparing in lower case catches both.

In [ ]:
coupons = []
types = []
years = []
categories = []

for name in df["name"]:
    coupon_text, rest = name.split("%")
    rest = rest.strip()

    coupons.append(coupon_to_number(coupon_text))
    years.append(int(rest[-4:]))
    types.append(rest[:-4].strip())

    if "index-linked" in name.lower():
        categories.append("Index-linked")
    elif "green" in name.lower():
        categories.append("Green")
    else:
        categories.append("Conventional")

df["coupon"] = coupons
df["type"] = types
df["redemptionYear"] = years
df["category"] = categories

df[["name", "coupon", "type", "redemptionYear", "category"]]

<br>

### Always check your results

`value_counts()` lists every different value in a column, and how often it appears. It's a quick way to spot problems:

In [ ]:
df["type"].value_counts()

<br>

There's the `Index-Linked` gilt again: it counts as a separate type. We fix it by replacing the text:

In [ ]:
df["type"] = df["type"].str.replace("Index-Linked", "Index-linked")
df["type"].value_counts()

<br><br>

---

<br>

## 6. Numbers and dates

Two more columns need cleaning:
- **Amounts** are text with commas (`"33,824"`). Remove the commas, then convert to numbers. These are £ million.
- **Dates** are text with a space at the start (`" 31-Jan-2023"`). `.str.strip()` removes it, then `pd.to_datetime()` turns the text into real dates. `format="%d-%b-%Y"` describes the pattern: day, short month name, 4-digit year.

In [ ]:
df["amount"] = pd.to_numeric(df["amount"].str.replace(",", ""))

for column in ["redemptionDate", "firstIssueDate"]:       # the same steps for both date columns
    df[column] = pd.to_datetime(df[column].str.strip(), format="%d-%b-%Y")

df[["name", "amount", "redemptionDate", "firstIssueDate"]].head()

<br><br>

---

<br>

## 7. The clean table

Keep the clean columns, check them, and save them as a CSV file.

In [ ]:
clean = df[["code", "name", "category", "type", "coupon", "redemptionYear",
            "redemptionDate", "firstIssueDate", "amount", "section"]]

clean.info()        # each column, how many values it has, and its type (numbers, dates or text)

In [ ]:
clean.to_csv("GiltsInIssueClean.csv", index=False)
clean.sample(5)     # 5 rows picked at random

<br><br>

---

<br>

## 8. Chart it

With clean columns, a chart is easy. When does the UK government need to repay its debt?

In [ ]:
chart = alt.Chart(
    clean,
    title=alt.Title("When is UK government debt repaid?",
                    subtitle="Gilts in issue on 20 October 2022, £ billion (nominal; index-linked gilts exclude inflation uplift). Source: DMO",
                    anchor="start", subtitleFontSize=10, subtitleColor="grey"),
    width=500, height=250,
).transform_calculate(
    amount_bn="datum.amount / 1000"
).mark_bar(size=7).encode(           # size: the width of each bar, in pixels
    x=alt.X("redemptionYear:Q").title(None).axis(format="d", grid=False),   # format "d": 2030, not 2,030
    y=alt.Y("sum(amount_bn):Q").title(None),
    color=alt.Color("category:N").title(None).scale(domain=["Conventional", "Index-linked", "Green"],
                                                    range=["#58b3c7", "#f4a582", "#2e8b57"]),
    tooltip=["redemptionYear:Q", "category:N", alt.Tooltip("sum(amount_bn):Q").format(".1f").title("£ billion")],
).configure_view(stroke="transparent")

chart.save("gilts_by_redemption_year.json")
chart

<br>

Download the clean data and the chart (Colab only):

In [ ]:
from google.colab import files     # only exists in Colab

for filename in ["GiltsInIssueClean.csv", "gilts_by_redemption_year.json"]:
    files.download(filename)

<br><br>

---

<br>

## <font color='green'>Try it</font>

**1.** Add a column `yearsLeft`: the number of years from 2022 until each gilt is repaid. Then use a loop with `if`/`else` to label each gilt `"Under 10 years"` or `"10 years or more"`.

**2.** Which gilt pays the highest coupon? Hint: `clean.sort_values("coupon")`.

**3.** What is the average coupon for each category? Hint: `clean.groupby("category")["coupon"].mean()`.

**4. (Stretch)** Download the latest list from the [DMO website](https://dmo.gov.uk/data/gilt-market/gilts-in-issue/) and run this notebook on it. What do you need to change? (Real data often changes format: this is why we check our results.)

In [ ]:
# Your code here


<br><br>

---

<br>

## Annex: the pandas shortcut

pandas has **string methods** that work on a whole column at once, with `.str`, so we don't need to write the loop ourselves (pandas runs it for us). These match the string functions on the lecture slide: split, substring, trim.

This gives the same coupon, type and year columns as section 5:

In [ ]:
parts = df["name"].str.split("%", expand=True)      # expand=True: one column for each piece
rest = parts[1].str.strip()

shortcut = pd.DataFrame({
    "coupon": parts[0].apply(coupon_to_number),        # .apply() runs our function on every row
    "type": rest.str[:-4].str.strip().str.replace("Index-Linked", "Index-linked"),
    "redemptionYear": rest.str[-4:].astype(int),
})

# Check: are the results the same as the loop's?
print(shortcut.equals(df[["coupon", "type", "redemptionYear"]]))
shortcut.head()